# Topic modelling

This notebook sets up a two-stage BERTopic modelling pipeline, including embedding calculations and a hyperparameter grid search.

______________________
# Importing packages

In [3]:
# config
from config import *

# Standard Library
import gc
import os
import pickle
import requests
import time
from itertools import product
from pathlib import Path
from time import sleep

# Third-Party Libraries
import hdbscan
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# Scikit-learn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score

# UMAP & Embeddings & BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

_____________________
# Preparing BERTopic

- Loading documents `docs_bert` that were created during data preparation.
- Creating a random (representative) sub-set of the data

In [2]:
# Reduce thread contention between libraries
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [4]:
# Loading documents
with open(FINAL_DOCS_PATH, "rb") as f:
    docs_bert = pickle.load(f)

print(f"Nr. of documents: {len(docs_bert)}")

Nr. of documents: 5332704


In [5]:
# Setting a random seed
np.random.seed(1)

# Create a subset of the data using 50,000 random documents
sample_idx = np.random.choice(len(docs_bert), 50_000, replace=False)
docs_sample = [docs_bert[i] for i in sample_idx]

Setting up helper functions used in later steps.

In [6]:
# Function for safely loading checkpoint pickle files
def safe_load_checkpoint(checkpoint_file):
    """Safely load checkpoint, handling corrupted/incomplete files."""
    if not os.path.exists(checkpoint_file):
        return None
    try:
        with open(checkpoint_file, 'rb') as f:
            checkpoint = pickle.load(f)
        return checkpoint
    except (EOFError, pickle.UnpicklingError):
        print("⚠️ Corrupted or incomplete checkpoint detected. Starting fresh...")
        os.remove(checkpoint_file)
        return None

In [7]:
# Function to access locally stored embedding models
def get_local_model_path(repo_id):
    cache_base = Path.home() / ".cache" / "huggingface" / "hub"
    cache_name = f"models--{repo_id.replace('/', '--')}"
    return cache_base / cache_name / "snapshots" / "main"

____________
# Running BERTopic

## Step 0: Compare Embedding Models
Run this section first to pick an embedding model that:

- Runs fast enough
- Produces stable embeddings (consistent cosine distances)
- Fits in memory
- Gives meaningful clusters in BERTopic with default settings, based on silhouette score

In [9]:
# Check if GPU is available
print(torch.cuda.is_available())  # should be True
print(torch.cuda.device_count())  # number of GPUs detected
print(torch.cuda.get_device_name(0))  # name of your GPU

True
1
NVIDIA GeForce RTX 3070 Ti


Ten embedding models are compared on a subset of our data using **BERTopic** with default settings to identify the best fit.

| Model | Source |
|:------|:--------|
| all-MiniLM-L6-v2 | [Hugging Face](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) |
| all-MiniLM-L12-v2 | [Hugging Face](https://huggingface.co/sentence-transformers/all-MiniLM-L12-v2) |
| all-mpnet-base-v2 | [SBERT Docs](https://www.sbert.net/docs/sentence_transformer/pretrained_models.html) |
| paraphrase-MiniLM-L6-v2 | [Hugging Face](https://huggingface.co/sentence-transformers/paraphrase-MiniLM-L6-v2) |
| multi-qa-MiniLM-L6-cos-v1 | [Hugging Face](https://huggingface.co/sentence-transformers/multi-qa-MiniLM-L6-cos-v1) |
| paraphrase-multilingual-MiniLM-L12-v2 | [Hugging Face](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) |
| bge-small-en-v1.5 | [BAAI](https://huggingface.co/BAAI/bge-small-en-v1.5) |
| bge-base-en-v1.5 | [BAAI](https://huggingface.co/BAAI/bge-base-en-v1.5) |
| e5-small-v2 | [Intfloat](https://huggingface.co/intfloat/e5-small-v2) |
| e5-base-v2 | [Intfloat](https://huggingface.co/intfloat/e5-base-v2) |


In [10]:
test_models = [
    # Baseline models
    "sentence-transformers/all-MiniLM-L6-v2",  # fast baseline
    "sentence-transformers/all-MiniLM-L12-v2", # deeper, slower
    "sentence-transformers/all-mpnet-base-v2", # better quality

    # Specialized for short text & social media
    "sentence-transformers/paraphrase-MiniLM-L6-v2",  # optimized for short text
    "sentence-transformers/multi-qa-MiniLM-L6-cos-v1",  # query-focused
    
    # Multilingual
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",  # 50+ languages
    
    # Modern/larger models (if quality > speed)
    "BAAI/bge-small-en-v1.5",  # strong recent model, small size
    "BAAI/bge-base-en-v1.5",   # better quality version
    
    # E5 family (state-of-art open models)
    "intfloat/e5-small-v2",    # prefix with "query:" or "passage:"
    "intfloat/e5-base-v2",     # stronger version
]

### Downloading the embedding models

This function:

- Creates a local cache directory for a Hugging Face model.
- Downloads core model files (.bin, .safetensors, configs, tokenizer files).
- Downloads numbered submodule directories with their configs.
- Handles retries and skips existing files.
- Returns whether any files were successfully downloaded.

In [ ]:
def download_model_files(repo_id, cache_base=None, verbose=False, max_retries=3):
    """
    Robust downloader for sentence-transformer models.
    Downloads main config files, model weights, and numbered submodule directories (e.g., 0_Transformer/).
    Automatically retries large files and checks for both 'model.safetensors' and 'pytorch_model.bin'.

    Args:
    - repo_id: Identifier of the hugging face model repo, e.g., "sentence-transformers/all-MiniLM-L6-v2"
    - cache_base: optional local directory to store downloaded files. Defaults to ~/.cache/huggingface/hub
    - verbose: whether to print detailed download messages
    - max_retries: number of times to retry a download if it fails
    """

    # Set up Cache directory
    if cache_base is None:
        cache_base = Path.home() / ".cache" / "huggingface" / "hub"

    cache_name = f"models--{repo_id.replace('/', '--')}"
    snapshot_dir = cache_base / cache_name / "snapshots" / "main"
    snapshot_dir.mkdir(parents=True, exist_ok=True)

    # Base URL for downloading
    base_url = f"https://huggingface.co/{repo_id}/resolve/main/"

    # Core files to download, including model weights
    base_files = [
        "model.safetensors", "pytorch_model.bin", "config.json", "tokenizer.json",
        "tokenizer_config.json", "vocab.txt", "special_tokens_map.json",
        "sentence_bert_config.json", "config_sentence_transformers.json",
        "modules.json",
    ]

    # Initialize counters
    success = 0
    failed_files = []

    # Helper function to download a single file
    def download_file(url, output_path):
        for attempt in range(max_retries):
            try:
                with requests.get(url, stream=True, timeout=120) as r:
                    if r.status_code == 404:
                        return False
                    r.raise_for_status()
                    with open(output_path, "wb") as f:
                        for chunk in r.iter_content(chunk_size=8192):
                            if chunk:
                                f.write(chunk)
                return True
            except Exception as e:
                if verbose:
                    print(f"⚠️ Attempt {attempt+1} failed for {output_path.name}: {e}")
                sleep(5)
        return False

    # Download top-level files
    for filename in base_files:
        output_path = snapshot_dir / filename
        if output_path.exists() and output_path.stat().st_size > 0:
            success += 1
            continue
        url = base_url + filename
        if download_file(url, output_path):
            success += 1
        else:
            failed_files.append(filename)

    # Download numbered submodules (0_Transformer, 1_Pooling, etc.)
    for i in range(6):
        for module_type in ["Transformer", "Pooling", "Dense", "Normalization"]:
            subdir = f"{i}_{module_type}"
            subdir_url = base_url + subdir + "/config.json"
            subdir_path = snapshot_dir / subdir
            subdir_path.mkdir(exist_ok=True)
            cfg_file = subdir_path / "config.json"

            if cfg_file.exists():
                continue

            if download_file(subdir_url, cfg_file):
                success += 1

    # Verbose logging
    if verbose:
        print(f"✅ {repo_id}: {success} files downloaded into {snapshot_dir}")
        if failed_files:
            print(f"⚠️ Failed to download: {failed_files}")

    return success > 0

Run `download_model_files`.

In [ ]:
if __name__ == "__main__":

    # List of all models to download (loaded above)
    models_to_download = test_models

    print("🚀 Starting bulk model download...")
    cache_dir = Path.home() / ".cache" / "huggingface" / "hub"
    print(f"📁 Cache directory: {cache_dir}")
    print(f"📊 Models to download: {len(models_to_download)}\n")

    successful, failed = 0, []

    # Download loop
    for model in tqdm(models_to_download, desc="📦 Downloading models"):
        try:
            ok = download_model_files(model, verbose=False)
            if ok:
                successful += 1
            else:
                failed.append(model)
        except KeyboardInterrupt:
            print("\n⚠️ Interrupted by user — progress saved.")
            break
        except Exception:
            failed.append(model)

    # Summary report
    print("\n" + "=" * 60)
    print("📊 DOWNLOAD SUMMARY")
    print("=" * 60)
    print(f"✅ Successful: {successful}/{len(models_to_download)}")
    if failed:
        print(f"❌ Failed: {len(failed)}")
        for m in failed:
            print(f"   - {m}")

### Comparing different embedding models

This function evaluates multiple SentenceTransformer models on a list of documents (`docs`) by embedding the documents, clustering them using BERTopic, and then computing clustering metrics (silhouette score, topic diversity, etc.). The output is a pandas DataFrame summarizing the performance of each model.

In [ ]:
def evaluate_sentence_transformers(
    docs,
    model_names,
    batch_size=128,
    verbose=True
):
    """
    Args: 
    - docs: a list of text documents to analyze.
    - model_names: a list of SentenceTransformer model names to evaluate.
    - batch_size: batch size used when computing embeddings.
    - verbose: whether to print progress messages.
    """

    # Device set up
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if verbose:
        print("Using device:", device)

    results = []

    # Loop over models
    for model_name in model_names:
        if verbose and torch.cuda.is_available():
            print(f"\nGPU Memory before {model_name}: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        print(f"\n🔹 Evaluating {model_name}...")

        start_time = time.time()

        # Load sentence transformer model
        try:
            local_path = get_local_model_path(model_name)
            if local_path.exists():
                model = SentenceTransformer(str(local_path), device=device)
            else:
                model = SentenceTransformer(model_name, device=device)
        except Exception as e:
            print(f"⚠️ Error with {model_name}, falling back to CPU: {e}")
            model = SentenceTransformer(model_name, device="cpu")

        # Compute embeddings
        embeddings = model.encode(
            docs,
            batch_size=batch_size,
            show_progress_bar=True,  # shows tqdm per model
            device=device
        )

        # Fit BERTopic
        topic_model = BERTopic(embedding_model=None, verbose=False)
        topics, _ = topic_model.fit_transform(docs, embeddings)

        # Computer metrics
        unique_topics = len(set(topics))
        silhouette = silhouette_score(embeddings, topics) if unique_topics > 1 else -1
        diversity = topic_model.get_topic_info().shape[0]
        duration = round(time.time() - start_time, 2)

        # Store results
        results.append({
            "model": model_name,
            "num_topics": unique_topics,
            "silhouette": silhouette,
            "topic_diversity": diversity,
            "time_seconds": duration
        })

        if verbose:
            print(f"✅ Finished {model_name}: topics={unique_topics}, silhouette={silhouette:.4f}, "
                  f"diversity={diversity}, time={duration}s")

        # Cleanup
        del model, embeddings, topic_model
        torch.cuda.empty_cache()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.ipc_collect()

    return pd.DataFrame(results).sort_values(by="silhouette", ascending=False)

In [ ]:
# Run the function on docs_sample
embedtest_results_df = evaluate_sentence_transformers(docs_sample, test_models, batch_size=128)

In [ ]:
print("\nFinal Results:")
embedtest_results_df

In [ ]:
# Save results
embedtest_results_df.to_pickle(EMBED_TEST_PATH)

## Step 1: Creating Embeddings

Selecting the best model based on Step 0:

In [ ]:
# Reload results from Step 0
embedtest_results_df = pd.read_pickle(EMBED_TEST_PATH)

best_model_name = embedtest_results_df.iloc[0]["model"]
print("🏆 Best embedding model:", best_model_name)

🏆 Best embedding model: sentence-transformers/all-MiniLM-L12-v2


### `encode_with_checkpointing`
This function:

- Embeds a large list of text documents using a SentenceTransformer model.
- Supports checkpointing so progress can be saved and resumed.
- Uses FP16 and PyTorch optimizations for speed on GPUs.
- Efficiently handles memory via pre-allocation or incremental storage.

In [ ]:
def encode_with_checkpointing(
    docs,
    model_name,
    batch_size=512,
    checkpoint_dir=TOPIC_MODELLING_FILES_DIR,
    checkpoint_every=5000,
    device=None,
    num_workers=4,
    use_fp16=True
):
    """
    Optimized encoding with checkpointing, recovery, and performance tuning.

    Args:
    - docs: List of text documents to embed
    - model_name: SentenceTransformer model name
    - batch_size: Batch size for encoding
    - checkpoint_dir: Directory to save checkpoints
    - checkpoint_every: Save checkpoint after this many docs
    - device: 'cuda', 'cpu', or None for auto
    - num_workers: Data loading workers
    - use_fp16: Use half precision for speed (GPU only)

    Returns:
        final_embeddings.npy: Numpy array of embeddings
    """
    
    # Prepare checkpointing paths
    os.makedirs(checkpoint_dir, exist_ok=True)
    checkpoint_file = os.path.join(checkpoint_dir, "checkpoint_embeddings.pkl")
    final_file = os.path.join(checkpoint_dir, "final_embeddings.npy")

    # Load existing embeddings if available
    if os.path.exists(final_file):
        print("✅ Loading existing embeddings...")
        return np.load(final_file)

    # Auto-detect device
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # Initialize SentenceTransformer model
    try:
            local_path = get_local_model_path(model_name)
            if local_path.exists():
                embedder = SentenceTransformer(str(local_path), device=device)
            else:
                embedder = SentenceTransformer(model_name, device=device)
    except Exception as e:
            print(f"⚠️ Error with {model_name}, falling back to CPU: {e}")
            embedder = SentenceTransformer(model_name, device="cpu")

    # Performance optimizations
    if device == "cuda" and use_fp16:
        embedder.half()
        print("FP16 mixed precision enabled")

    if hasattr(torch, "compile") and device == "cuda":
        try:
            embedder = torch.compile(embedder)
            print("Model compiled with PyTorch 2.0+ optimization")
        except Exception:
            pass

    # Load previous checkpoint
    checkpoint = safe_load_checkpoint(checkpoint_file) # Helper function defined earlier
    start_idx, embeddings_list, all_embeddings = 0, [], None

    # Resume from last saved document
    if checkpoint:
        start_idx = checkpoint.get("last_idx", 0)
        embeddings_list = checkpoint.get("embeddings", [])
        if "partial_array" in checkpoint and checkpoint["partial_array"] is not None:
            all_embeddings = checkpoint["partial_array"]
        print(f"Resuming from document {start_idx}/{len(docs)}")

    # Preallocate memory
    use_preallocated = False
    if all_embeddings is None and len(docs) < 10_000_000:
        try:
            embedding_dim = embedder.get_sentence_embedding_dimension()
            all_embeddings = np.empty((len(docs), embedding_dim), dtype=np.float32)
            use_preallocated = True
            print(f"Pre-allocated embedding array: {all_embeddings.shape}")
        except Exception:
            print("Memory pre-allocation skipped (insufficient memory).")

    # Main encoding loop
    try:
        batches = range(start_idx, len(docs), batch_size)
        pbar = tqdm(
            batches,
            desc="Encoding",
            initial=start_idx // batch_size,
            total=(len(docs) + batch_size - 1) // batch_size,
        )

        for i in pbar:
            batch = docs[i:min(i + batch_size, len(docs))]

            # Encode batch efficiently
            with torch.no_grad():
                batch_embeddings = embedder.encode(
                    batch,
                    batch_size=batch_size,
                    show_progress_bar=False,
                    convert_to_numpy=True,
                    normalize_embeddings=False,
                    device=device,
                )

            # Store embeddings
            if use_preallocated:
                end_idx = min(i + batch_size, len(docs))
                all_embeddings[i:end_idx] = batch_embeddings
            else:
                embeddings_list.append(batch_embeddings)

            # Progress update
            if i > start_idx and pbar.format_dict['elapsed'] > 0:
                docs_per_sec = (i - start_idx) / pbar.format_dict['elapsed']
                pbar.set_postfix({'docs/sec': f'{docs_per_sec:.0f}'})

            # Periodic checkpointing
            if (i + batch_size - start_idx) % checkpoint_every == 0 or i + batch_size >= len(docs):
                checkpoint = {
                    'last_idx': min(i + batch_size, len(docs)),
                    'embeddings': embeddings_list if not use_preallocated else None,
                    'partial_array': all_embeddings[:min(i + batch_size, len(docs))] if use_preallocated else None
                }
                with open(checkpoint_file, 'wb') as f:
                    pickle.dump(checkpoint, f, protocol=pickle.HIGHEST_PROTOCOL)

    # Handle interruptions
    except KeyboardInterrupt:
        print("\n⏸ Interrupted! Saving progress...")
        checkpoint = {
            'last_idx': i,
            'embeddings': embeddings_list if not use_preallocated else None,
            'partial_array': all_embeddings[:i] if use_preallocated else None,
        }
        with open(checkpoint_file, 'wb') as f:
            pickle.dump(checkpoint, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"Checkpoint saved at document {i}. Restart to resume.")
        raise

    # Combine embeddings
    embeddings = all_embeddings if use_preallocated else np.vstack(embeddings_list)

    # Save final embeddings
    np.save(final_file, embeddings)
    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)

    print(f"✅ Encoding complete! Shape: {embeddings.shape}")
    return embeddings

Run the cell below to create embeddings. This will run fastest on a GPU.

In [ ]:
# Create embeddings
embeddings = encode_with_checkpointing(docs_bert, model_name=best_model_name)

## Step 2: Gridsearch

Please note that the `docs` input file needs to be exactly the same one that is used in Step 1 to compute embeddings.

Reloading the embeddings created in step 1 to skip `encoding_with_checkpointing`.

In [ ]:
# Reload embeddings to skip step 1
embeddings = np.load(
    EMBEDDINGS_PATH,
    allow_pickle=True
)

# Check shape
print("Type:", type(embeddings))
print("Shape:", getattr(embeddings, "shape", None))

### Parameter Tuning

A grid search is used to identify optimal parameters before scaling to the full dataset. Parameter definitions are retrieved from [Grootendorst (2025)](https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html#n_neighbors).

---

### UMAP Parameters

#### n_neighbors
**Definition:** The number of neighboring sample points used when making the manifold approximation. Increasing this value results in a more global view and larger clusters, while smaller values result in a more local view ([McInnes, Healy, & Melville, 2018](https://arxiv.org/abs/1802.03426)).

**Tested values:**
- **10** — Captures local structures by grouping videos with nearly identical hashtag combinations
- **15** — UMAP default that balances local vs. global patterns; works well for medium-sized granularity
- **30** — Captures broader patterns to connect videos across wider semantic spaces

#### n_components
**Definition:** The dimensionality of embeddings after reduction. Lower values can hurt embedding quality; higher values make HDBSCAN clustering more difficult. Very high dimensionality requires metrics suited for high-dimensional data and increases runtime ([Grootendorst, 2022](https://arxiv.org/abs/2203.05794); [McInnes et al., 2018](https://arxiv.org/abs/1802.03426)).

**Tested values:**
- **5** — Minimal dimensionality; assumes simple hashtag semantic space with few core themes; fast processing
- **10** — Moderate complexity; allows for more nuanced topic distinctions
- **15** — High dimensionality; preserves maximum information from embeddings and catches subtle differences between similar hashtags

#### min_dist
**Definition:** The minimum distance between points in UMAP space. Values closer to 0 create more compact clusters. Increase for looser clusters, decrease for tighter ones ([McInnes et al., 2018](https://arxiv.org/abs/1802.03426)).

**Tested values:**
- **0.0** — Tightest possible clusters
- **0.1** — Slightly looser; strikes a good balance for overlapping hashtag communities
- **0.3** — Loose clusters that maintain global structure when hashtag categories have fuzzy boundaries (e.g., #fitness overlaps with #health, #workout)

---

### HDBSCAN Parameters

#### min_cluster_size
**Definition:** Controls the minimum number of documents needed to form a cluster, thereby affecting the total number of clusters generated. The default is 10. Increasing this value produces fewer but larger clusters; decreasing it generates more micro-clusters. Generally advised to increase rather than decrease ([Grootendorst, 2025](https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html#n_neighbors); [McInnes & Healy, 2017](https://arxiv.org/abs/1705.07321)).

**Tested values:**
- **10** — Allows micro-topics to capture niche communities; finds rare but coherent hashtag patterns
- **30** — Filters out the tiniest niches; focuses on slightly more popular hashtags to reduce noise while keeping specificity
- **50** — Finds established hashtag trends that appear consistently
- **100** — Captures major themes; focuses on dominant patterns while reducing total topic count

#### min_samples
**Definition:** Controls the number of outliers generated; automatically set to match `min_cluster_size` by default. Setting this significantly lower than `min_cluster_size` may reduce noise, but outliers are expected and forcing zero outliers may misrepresent the data ([Campello, Moulavi, & Sander, 2013](https://link.springer.com/chapter/10.1007/978-3-642-37456-2_14#citeas)).

**Tested values:**
- **15** — Liberal clustering; videos need only 15 similar neighbors to form a cluster core; creates more topics to catch emerging hashtag trends early
- **25** — Moderately strict; requires stronger evidence of cluster membership; balances capturing trends and avoiding noise
- **35** — Conservative approach; only forms clusters with strong consensus; results in fewer, high-confidence topics

In [12]:
# Take a sub-set of the embeddings for grid search
embeddings_sample = embeddings[sample_idx].copy()

In [ ]:
# Function to do BERTopic grid search with checkpointing
def bertopic_grid_search(
    docs,
    embeddings,
    param_grid,
    checkpoint_path=CHECKPOINT_PATH_GRIDSEARCH,
    save_csv_path=FINAL_CSV_PATH_GRIDSEARCH,
    verbose=True
):
    """
    Perform grid search over BERTopic UMAP and HDBSCAN parameters with checkpointing.

    Args:
        docs: List of documents
        embeddings: Precomputed embeddings for the documents
        param_grid: Dictionary of parameters to search over
        checkpoint_path: Path to save/load checkpoint
        save_csv_path: Path to save final results CSV
        verbose: Whether to print progress

    Returns:
        grid_results_df: DataFrame with results sorted by silhouette score
        best_params: Dictionary of best parameter combination
    """
    start_time = time.time()
    
    # Load checkpoint if it exists
    results = []
    checkpoint = safe_load_checkpoint(checkpoint_path)
    if checkpoint:
        results = checkpoint.get("results", [])
        start_index = checkpoint.get("last_index", -1) + 1
        if verbose:
            print(f"✅ Loaded checkpoint — resuming from combination #{start_index}")
    else:
        start_index = 0

    # Create all parameter combinations
    param_combinations = list(product(
        param_grid["umap__n_neighbors"],
        param_grid["umap__n_components"],
        param_grid["umap__min_dist"],
        param_grid["hdbscan__min_cluster_size"],
        param_grid["hdbscan__min_samples"]
    ))

    total_combos = len(param_combinations)
    if verbose:
        print(f"🔍 Total combinations to test: {total_combos}")

    # Grid search loop
    for idx, (n_neighbors, n_components, min_dist, min_cluster_size, min_samples) in enumerate(param_combinations[start_index:], start=start_index):
        if verbose:
            print(f"\n🔹 Testing {idx+1}/{total_combos}: "
                  f"neighbors={n_neighbors}, comp={n_components}, dist={min_dist}, "
                  f"cluster={min_cluster_size}, samples={min_samples}")
        try:
            # UMAP
            umap_model = UMAP(
                n_neighbors=n_neighbors,
                n_components=n_components,
                min_dist=min_dist,
                metric='cosine',
                low_memory=True,
                random_state=42,
                n_jobs=-1
            )

            # HDBSCAN
            hdbscan_model = hdbscan.HDBSCAN(
                min_cluster_size=min_cluster_size,
                min_samples=min_samples,
                metric='euclidean',
                cluster_selection_method='eom',
                core_dist_n_jobs=-1,
                prediction_data=False
            )

            # CountVectorizer
            vectorizer_model = CountVectorizer(
                ngram_range=(1, 1),
                stop_words="english",
                max_features=10_000,
                min_df=1
            )

            # BERTopic
            topic_model = BERTopic(
                embedding_model=None,
                umap_model=umap_model,
                hdbscan_model=hdbscan_model,
                vectorizer_model=vectorizer_model,
                calculate_probabilities=False,
                verbose=False
            )

            # Fit model
            topics, _ = topic_model.fit_transform(docs, embeddings)

            # Metrics
            unique_topics = len(set(topics))
            silhouette = silhouette_score(embeddings, topics) if unique_topics > 1 else -1

            # Store results
            results.append({
                "n_neighbors": n_neighbors,
                "n_components": n_components,
                "min_dist": min_dist,
                "min_cluster_size": min_cluster_size,
                "min_samples": min_samples,
                "silhouette": silhouette,
                "num_topics": unique_topics
            })

            if verbose:
                print(f"✅ Done | Topics: {unique_topics} | Silhouette: {silhouette:.4f}")

        # Handling errors
        except Exception as e:
            if verbose:
                print(f"❌ Failed for combination: {e}")
            results.append({
                "n_neighbors": n_neighbors,
                "n_components": n_components,
                "min_dist": min_dist,
                "min_cluster_size": min_cluster_size,
                "min_samples": min_samples,
                "silhouette": np.nan,
                "num_topics": np.nan
            })

        # Save checkpoint
        checkpoint = {"results": results, "last_index": idx}
        with open(checkpoint_path, "wb") as f:
            pickle.dump(checkpoint, f, protocol=pickle.HIGHEST_PROTOCOL)
        if verbose:
            print("💾 Checkpoint saved.")

    # Final results
    grid_results_df = pd.DataFrame(results).sort_values(by="silhouette", ascending=False)
    best_params = grid_results_df.iloc[0].to_dict()

    # Save CSV
    grid_results_df.to_csv(save_csv_path, index=False)

    # Measure time
    elapsed = time.time() - start_time
    if verbose:
        print(f"\n🏁 Grid search complete in {elapsed/60:.1f} minutes")
        print(f"\nBest configuration:\n{best_params}")

    return grid_results_df, best_params

In [ ]:
# Define parameters
param_grid = {
    "umap__n_neighbors": [10, 15, 30],
    "umap__n_components": [5, 10, 15],
    "umap__min_dist": [0.0, 0.1, 0.3],
    "hdbscan__min_cluster_size": [10, 30, 50, 100],
    "hdbscan__min_samples": [15, 25, 35]
}

Using `docs_sample` and `embeddings_sample`. Please note that running the following cell will test 324 combinations, and requires several hours of runtime.

In [ ]:
grid_results_df, best_params = bertopic_grid_search(
    docs=docs_sample,
    embeddings=embeddings_sample,
    param_grid=param_grid,
    checkpoint_path=CHECKPOINT_PATH_GRIDSEARCH,
    save_csv_path=FINAL_CSV_PATH_GRIDSEARCH,
    verbose=True
)

🔍 Total combinations to test: 324

🔹 Testing 1/324: neighbors=10, comp=5, dist=0.0, cluster=10, samples=15
✅ Done | Topics: 639 | Silhouette: 0.0045
💾 Checkpoint saved.

🔹 Testing 2/324: neighbors=10, comp=5, dist=0.0, cluster=10, samples=25
✅ Done | Topics: 416 | Silhouette: 0.0055
💾 Checkpoint saved.

🔹 Testing 3/324: neighbors=10, comp=5, dist=0.0, cluster=10, samples=35
✅ Done | Topics: 306 | Silhouette: 0.0055
💾 Checkpoint saved.

🔹 Testing 4/324: neighbors=10, comp=5, dist=0.0, cluster=30, samples=15
✅ Done | Topics: 355 | Silhouette: 0.0120
💾 Checkpoint saved.

🔹 Testing 5/324: neighbors=10, comp=5, dist=0.0, cluster=30, samples=25
✅ Done | Topics: 337 | Silhouette: 0.0113
💾 Checkpoint saved.

🔹 Testing 6/324: neighbors=10, comp=5, dist=0.0, cluster=30, samples=35
✅ Done | Topics: 278 | Silhouette: 0.0111
💾 Checkpoint saved.

🔹 Testing 7/324: neighbors=10, comp=5, dist=0.0, cluster=50, samples=15
✅ Done | Topics: 216 | Silhouette: 0.0157
💾 Checkpoint saved.

🔹 Testing 8/324: nei

In [ ]:
grid_results_df

,n_neighbors,n_components,min_dist,min_cluster_size,min_samples,silhouette,num_topics
6,10,5,0.0,50,15,0.019309,218
294,30,15,0.0,50,15,0.018858,192
7,10,5,0.0,50,25,0.018669,210
256,30,10,0.0,30,25,0.018572,284
82,10,15,0.0,100,25,0.018400,107
...,...,...,...,...,...,...,...
60,10,10,0.3,10,15,-0.008738,462
97,10,15,0.3,10,25,-0.009519,296
61,10,10,0.3,10,25,-0.011161,298
24,10,5,0.3,10,15,-0.012070,471


## Step 3: BERTopic on the full data

In [31]:
# Reload grid search results
grid_results_df = pd.read_csv(FINAL_CSV_PATH_GRIDSEARCH)

In [32]:
# Get best parameters
best_params = grid_results_df.iloc[0].to_dict()
print(best_params)

{'n_neighbors': 10.0, 'n_components': 5.0, 'min_dist': 0.0, 'min_cluster_size': 50.0, 'min_samples': 15.0, 'silhouette': 0.0193086043000221, 'num_topics': 218.0}


### `fit_bertopic_online`

The function trains on a randomly sampled subset (training_sample_size) to discover topics, then assigns topics to remaining documents in batches. This approach is 
optimized for large datasets of short documents (e.g., hashtags) where topic saturation occurs early.

##### 1. Data Shuffling
First, the data is shuffled using a random permutation to ensure the training sample is representative of the full corpus. The function works on copies of the input data to preserve the original document order.

##### 2. Model Initialization
Initializes UMAP ([McInnes, Healy, & Melville, 2018](https://arxiv.org/abs/1802.03426)), HDBSCAN ([Campello et al., 2013](https://link.springer.com/chapter/10.1007/978-3-642-37456-2_14#citeas)), CountVectorizer, and BERTopic using the optimal parameters identified from the previous grid search.

##### 3. Training Phase
The training phase discovers the initial topic structure from a representative subset. Topics are discovered via `.fit_transform()` on a random subset of 50,000 documents by default. 

This training sample size was chosen because: 
- The hyperparameters from the [grid search](#step-2-gridsearch-for-umap--hdbscan-parameters) were optimized on this sample size
- It provides computational efficiency by not loading all 5.3M documents into UMAP/HDBSCAN simultaneously. 

While short text documents often lead to lower topic model performance ([Qiang, Qian, Li, Yuan, & Wu, 2020](https://ieeexplore.ieee.org/document/9086136)), hashtags are usually used in a way that a few popular tags often dominate usage ([Cunha, Magno, Comarela, Almeida, Gonçalves, & Benevenuto, 2011](https://dl.acm.org/doi/10.5555/2021109.2021117)). 

This approach assumes that the most common hashtags, biggest communities and major themes will appear in a small sub-sample. If topic saturation is not achieved (indicated by a high outlier rate of >20%), a subsequent run will be performed on all of the outliers.

##### 4. Prediction Phase
The remaining ~5.3 million documents are assigned to the discovered topics via `.transform()` in batches of 50,000 documents. Batch processing ensures memory efficiency when handling large-scale datasets ([Grootendorst, 2022](https://arxiv.org/abs/2203.05794)). 

After all predictions are complete, topics are restored to the original document order.

In [ ]:
def fit_bertopic(docs, embeddings, best_params, batch_size=50_000, 
                        training_sample_size=50_000, random_seed=42):
    
    """
     Fit BERTopic in batches to handle large datasets with tuned hyperparameters.
     
    Args:
        docs (list): List of documents.
        embeddings (np.ndarray): Pre-computed embeddings array.
        best_params (dict): Dictionary with best UMAP and HDBSCAN hyperparameters.
        batch_size (int): Number of documents to process per batch during prediction.
        training_sample_size (int): Number of documents to use for initial training (default: 50k).
        random_seed (int): Random seed for reproducible shuffling (default: 42).
    
    Returns:
        topic_model (BERTopic): Trained BERTopic model.
        topics (np.ndarray): Topic assignments in ORIGINAL document order.
    """

    
    # Generate shuffle indices only
    print(f"Preparing {len(docs):,} documents for representative sampling...")
    start_time = time.time()  # Add timing
    
    np.random.seed(random_seed) # random_seed defined earlier
    shuffle_indices = np.random.permutation(len(docs))
    print(f"  ✓ Indices generated in {time.time() - start_time:.2f}s")
    
    # Only shuffle embeddings
    print(f"Shuffling embeddings array ({embeddings.nbytes / 1e9:.2f} GB)...")
    start_time = time.time()
    
    embeddings_shuffled = embeddings[shuffle_indices].copy()
    
    elapsed = time.time() - start_time
    print(f"  ✓ Embeddings shuffled in {elapsed:.2f}s ({embeddings.nbytes / 1e9 / elapsed:.2f} GB/s)")
    
    # Determine actual training size
    actual_training_size = min(training_sample_size, len(docs))
    print(f"Training on {actual_training_size:,} documents ({actual_training_size/len(docs)*100:.1f}% of corpus)")
    
    # Initialize models using parameters from best_params
    umap_model = UMAP(
        n_neighbors=int(best_params["n_neighbors"]),
        n_components=int(best_params["n_components"]),
        min_dist=best_params["min_dist"],
        metric="cosine",    # Measures semantic similarity
        random_state=random_seed,
        n_jobs=-1,
        low_memory=True
    )

    hdbscan_model = HDBSCAN(
        min_cluster_size=int(best_params["min_cluster_size"]),
        min_samples=int(best_params["min_samples"]),
        metric="euclidean",
        cluster_selection_method="eom",
        core_dist_n_jobs=-1,
        prediction_data=True
    )

    # Set up CountVectorizer for BERT
    vectorizer_model = CountVectorizer(
        ngram_range=(1, 1), # Only consider single words (unigrams),as hashtags are individual tokens
        stop_words="english", # Most hashtags do not contain stopwords
        max_features=10_000, # Focus on the top 10,000 most frequent hashtags across the corpus for topic representation
        min_df=1 # Lenient: a hashtag only needs to appear in at least 1 document to be included
    )

    # BERTopic
    topic_model = BERTopic(
        embedding_model=None,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=False,
        verbose=True    # Print progress
    )

    # Training Phase - index docs on-the-fly
    print("\n🔧 Training model on sampled subset...")
    training_indices = shuffle_indices[:actual_training_size]
    training_docs = [docs[i] for i in training_indices]  # Fast: only 50k
    training_embeddings = embeddings_shuffled[:actual_training_size]

    topics_shuffled, _ = topic_model.fit_transform(training_docs, training_embeddings)
    print(f"✅ Training complete. Discovered {len(np.unique(topics_shuffled))} topics (including outliers)")

    # Prediction Phase - batch-by-batch indexing
    if len(docs) > actual_training_size:
        print(f"\n🔮 Assigning topics to remaining {len(docs) - actual_training_size:,} documents...")
        
        for i in tqdm(range(actual_training_size, len(docs), batch_size), 
                     desc="Processing batches", leave=False):
            end_idx = min(i + batch_size, len(docs))
            
            # Index docs on-the-fly for this batch only (50k lookups per batch)
            batch_indices = shuffle_indices[i:end_idx]
            batch_docs = [docs[j] for j in batch_indices]
            batch_embeddings = embeddings_shuffled[i:end_idx]

            batch_topics, _ = topic_model.transform(batch_docs, batch_embeddings)
            topics_shuffled = np.concatenate([topics_shuffled, batch_topics])

    # Restore original order
    topics = np.empty(len(topics_shuffled), dtype=topics_shuffled.dtype)
    topics[shuffle_indices] = topics_shuffled
    
    # Calculate metrics
    outlier_count = np.sum(topics == -1)
    outlier_rate = outlier_count / len(topics) * 100
    n_topics = len(np.unique(topics[topics != -1]))
    
    print(f"\n📊 Outlier rate: {outlier_rate:.2f}% ({outlier_count:,} documents)")
    print(f"→ Total unique topics: {n_topics}")
    print(f"→ Total documents processed: {len(topics):,}")
    
    return topic_model, topics

In [ ]:
# Verify everything is ready
print("Pre-flight checklist:")
print(f"✓ Documents loaded: {len(docs_bert):,}")
print(f"✓ Embeddings loaded: {embeddings.shape}")
print(f"✓ Embeddings match docs: {len(docs_bert) == embeddings.shape[0]}")
print(f"✓ Best params defined: {best_params}")
print(f"✓ Function imported: {'fit_bertopic' in dir()}")

Pre-flight checklist:
✓ Documents loaded: 5,332,704
✓ Embeddings loaded: (5332704, 384)
✓ Embeddings match docs: True
✓ Best params defined: {'n_neighbors': 10.0, 'n_components': 5.0, 'min_dist': 0.0, 'min_cluster_size': 50.0, 'min_samples': 15.0, 'silhouette': 0.0193086043000221, 'num_topics': 218.0}
✓ Function imported: True


In [ ]:
# Running BERTopic on docs_bert and embeddings
topic_model, topics = fit_bertopic(
    docs=docs_bert, 
    embeddings=embeddings,
    best_params=best_params,
    batch_size=50_000,
    training_sample_size=50_000
)

print(f"\nGenerated {len(topic_model.get_topic_info())} topics")
print(f"Assigned topics to {len(topics):,} documents")

# Validate topic quality
non_outlier_topics = topics[topics != -1]
print(f"Non-outlier documents: {len(non_outlier_topics):,} ({len(non_outlier_topics)/len(topics)*100:.1f}%)")

# Calculate Silhouette Score
print("\n📊 Calculating Silhouette Score...")

# Use a sample for computational efficiency
sample_size = 50_000
sample_indices = np.random.choice(len(topics), size=min(sample_size, len(topics)), replace=False)

sample_topics = topics[sample_indices]
sample_embeddings = embeddings[sample_indices]

# Remove outliers (topic -1) for silhouette calculation
non_outlier_mask = sample_topics != -1
sample_topics_clean = sample_topics[non_outlier_mask]
sample_embeddings_clean = sample_embeddings[non_outlier_mask]

# Calculate & print silhouette
silhouette = silhouette_score(sample_embeddings_clean, sample_topics_clean)
print(f"Silhouette Score (on {len(sample_topics_clean):,} sample): {silhouette:.4f}")


Save & load the model & topic assignments.

In [ ]:
# Save topic model
topic_model.save(TOPIC_MODEL_PATH)

# Save topics
np.save(TOPICS_ASSIGNED_PATH, topics)

## Step 4: BERTopic on outliers

Above 20% of our documents were classified as 'outliers'. As this is a large proportion of the total number of documents, these outliers may not have been assigned a topic not because they are "noise", but they may contain valid topics at a finer granularity. Hence, a second stage is performed on all outliers to discover niche topics.

In [ ]:
# Reload ORIGINAL topic model in case it is needed
topic_model = BERTopic.load(TOPIC_MODEL_PATH)
topics = np.load(TOPICS_ASSIGNED_PATH)

In [ ]:
# Select outlier documents
outlier_docs = [doc for doc, t in zip(docs_bert, topics) if t == -1]

In [ ]:
# Save outlier documents
with open(OUTLIER_DOCS_PATH, "wb") as f:
    pickle.dump(docs_bert, f)

In [ ]:
# Get indexes of outliers from original topic assignments
outlier_idx = [i for i, t in enumerate(topics) if t == -1]

# Select only the embeddings for outliers
outlier_embeddings = embeddings[outlier_idx]

# Just to confirm
print(outlier_embeddings.shape, len(outlier_docs))

Modelling outliers one more time.

In [ ]:
# Verify everything is ready
print("**OUTLIER TOPIC MODELLING*\n\nPre-flight checklist:")
print(f"✓ Documents loaded: {len(outlier_docs):,}")
print(f"✓ Embeddings loaded: {outlier_embeddings.shape}")
print(f"✓ Embeddings match docs: {len(outlier_docs) == outlier_embeddings.shape[0]}")
print(f"✓ Best params defined: {best_params}")
print(f"✓ Function imported: {'fit_bertopic_online' in dir()}")

**OUTLIER TOPIC MODELLING*

Pre-flight checklist:
✓ Documents loaded: 1,541,249
✓ Embeddings loaded: (1541249, 384)
✓ Embeddings match docs: True
✓ Best params defined: {'n_neighbors': 10.0, 'n_components': 5.0, 'min_dist': 0.0, 'min_cluster_size': 50.0, 'min_samples': 15.0, 'silhouette': 0.0193086043000221, 'num_topics': 218.0}
✓ Function imported: False


In [ ]:
# Run BERTopic on outliers
if __name__ == "__main__":
    topic_model_outliers, topics_outliers = fit_bertopic(
        docs=outlier_docs, 
        embeddings=outlier_embeddings,
        best_params=best_params,
        batch_size=50_000,
        training_sample_size=50_000,
        random_seed=43 # Changed from default 42 to prevent sampling bias
    )
    
    print(f"\nGenerated {len(topic_model_outliers.get_topic_info())} topics")
    print(f"Assigned topics to {len(topics_outliers):,} documents")
    
    # Validate topic quality
    non_outlier_o_topics = topics_outliers[topics_outliers != -1]
    print(f"Non-outlier documents: {len(non_outlier_o_topics):,} ({len(non_outlier_o_topics)/len(topics_outliers)*100:.1f}%)")
    
    # Calculate Silhouette Score
    print("\n📊 Calculating Silhouette Score...")
    
    # Use a new sample for computational efficiency
    sample_size = 50_000
    sample_indices = np.random.choice(len(topics_outliers), size=min(sample_size, len(topics_outliers)), replace=False)
    
    sample_topics = topics_outliers[sample_indices]
    sample_embeddings = outlier_embeddings[sample_indices]
    
    # Remove outliers (topic -1) for silhouette calculation
    non_outlier_mask = sample_topics != -1
    sample_topics_clean = sample_topics[non_outlier_mask]
    sample_embeddings_clean = sample_embeddings[non_outlier_mask]

    silhouette_outliers = silhouette_score(sample_embeddings_clean, sample_topics_clean)
    print(f"Silhouette Score (on {len(sample_topics_clean):,} sample): {silhouette_outliers:.4f}")

In [ ]:
# Save outlier topic model
topic_model_outliers.save(OUTLIER_MODEL_PATH)

# Save outlier topics
np.save(OUTLIER_TOPICS_PATH, topics_outliers)

__________________
# AI disclosure statement

AI tools were used to assist:
- developing, labelling, and debugging code
- formatting Markdown cells

AI tools used:
- [CursorAI (Desktop version)](https://cursor.com/agents)
- [ClaudeAI](https://claude.ai/)
- [ChatGPT](https://chatgpt.com/)

I acknowledge my responsibility as a researcher to thoroughly verify all outputs and content produced by AI tools and accept full accountability for their accuracy and validity.

Inga Vondenhof

_____________
# References

- Campello, R. J. G. B., Moulavi, D., & Sander, J. (2013). Density-based clustering based on hierarchical density estimates. In J. Pei, V. S. Tseng, L. Cao, H. Motoda, & G. Xu (Eds.), *Advances in Knowledge Discovery and Data Mining: PAKDD 2013* (Lecture Notes in Computer Science, Vol. 7819, pp. 160–172). Springer. https://doi.org/10.1007/978-3-642-37456-2_14

- Cunha, E., Magno, G., Comarela, G., Almeida, V., Gonçalves, M. A., & Benevenuto, F. (2011). Analyzing the dynamic evolution of hashtags on Twitter: A language-based approach. In *Proceedings of the Workshop on Languages in Social Media (LSM ’11)* (pp. 58–65). Association for Computational Linguistics. https://dl.acm.org/doi/10.5555/2021109.2021117

- Grootendorst, M. (2025). Parameter tuning — BERTopic documentation. Retrieved October 30, 2025, from https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html#prediction_data

- Grootendorst, M. (2022). *BERTopic: Neural topic modeling with a class-based TF-IDF procedure* (arXiv preprint arXiv:2203.05794). https://doi.org/10.48550/arXiv.2203.05794

- McInnes, L., Healy, J., & Melville, J. (2018). *UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction* (arXiv preprint arXiv:1802.03426). https://doi.org/10.48550/arXiv.1802.03426

- McInnes, L., & Healy, J. (2017). *Accelerated hierarchical density based clustering. In 2017 IEEE International Conference on Data Mining Workshops (ICDMW)* (pp. 33–42). IEEE. 
https://doi.org/10.48550/arXiv.1705.07321

- Qiang, J., Zhang, Y., Li, X., & Wang, H. (2022). Short text topic modeling techniques, applications, and performance: A survey. *IEEE Transactions on Knowledge and Data Engineering, 34*(3), 1427–1445. [10.1109/TKDE.2020.2992485](https://ieeexplore.ieee.org/document/9086136)